In [69]:
import cv2
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import os
import random
import torch
from torch.utils.data import DataLoader, TensorDataset,WeightedRandomSampler
from sklearn.model_selection import train_test_split
from torchvision import transforms
from PIL import Image

In [70]:
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch.nn as nn
import torch.nn.functional as F

In [71]:
import kagglehub

In [72]:
path=kagglehub.dataset_download("andrewmvd/leukemia-classification")

Using Colab cache for faster access to the 'leukemia-classification' dataset.


In [73]:
print(os.getcwd())

/content


In [74]:



# all -> leukemia positive, hem = normal
DATA_DIR = "../kaggle/input/leukemia-classification/C-NMC_Leukemia"
TRAIN_DIR = os.path.join(DATA_DIR,"training_data")
TEST_DIR = os.path.join(DATA_DIR,"testing_data")
VALIDATION_DIR = os.path.join(DATA_DIR,"validation_data")

def load_image(path):
    """Reads BGR image → RGB. Returns None if unreadable."""
    try:
        img = cv2.imread(path)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    except:
        return None


def get_folds(folder_path, max_workers=8):
    folds = []

    for fold_name in sorted(os.listdir(folder_path)):
        fold_path = os.path.join(folder_path, fold_name)

        all_path = os.path.join(fold_path, "all")
        hem_path = os.path.join(fold_path, "hem")

        # Build list of image paths
        all_images_paths = [os.path.join(all_path, f) for f in os.listdir(all_path)]
        hem_images_paths = [os.path.join(hem_path, f) for f in os.listdir(hem_path)]

        # --- Multithreading ---
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Submit tasks for ALL images
            all_futures = [executor.submit(load_image, p) for p in all_images_paths]
            hem_futures = [executor.submit(load_image, p) for p in hem_images_paths]

            # Collect completed results
            all_images = [f.result() for f in as_completed(all_futures) if f.result() is not None]
            hem_images = [f.result() for f in as_completed(hem_futures) if f.result() is not None]

        folds.append({
            "all": all_images,
            "hem": hem_images,
        })

    return folds
def get_data(folder_path,max_workers = 8):
  data = []
  labels = []
  for fold_name in sorted(os.listdir(folder_path)):
    fold_path = os.path.join(folder_path, fold_name)

    all_path = os.path.join(fold_path, "all")
    hem_path = os.path.join(fold_path, "hem")

    # Build list of image paths
    all_images_paths = [os.path.join(all_path, f) for f in os.listdir(all_path)]
    hem_images_paths = [os.path.join(hem_path, f) for f in os.listdir(hem_path)]

    # --- Multithreading ---
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit tasks for ALL images
        all_futures = [executor.submit(load_image, p) for p in all_images_paths]
        hem_futures = [executor.submit(load_image, p) for p in hem_images_paths]

        # Collect completed results
        all_images = [f.result() for f in as_completed(all_futures) if f.result() is not None]
        hem_images = [f.result() for f in as_completed(hem_futures) if f.result() is not None]

    data.extend(all_images)
    data.extend(hem_images)
    labels.extend([1]*len(all_images))
    labels.extend([0]*len(hem_images))
  return np.array(data),np.array(labels)

def get_paths(folder_path):
  all_paths = []
  hem_paths = []
  all_labels = []
  hem_labels = []

  for fold_name in sorted(os.listdir(folder_path)):
      fold_path = os.path.join(folder_path, fold_name)

      all_path = os.path.join(fold_path, "all")
      hem_path = os.path.join(fold_path, "hem")

      all_images_paths = [os.path.join(all_path, f) for f in os.listdir(all_path)]
      hem_images_paths = [os.path.join(hem_path, f) for f in os.listdir(hem_path)]

      all_paths.extend(all_images_paths)
      all_labels.extend([1] * len(all_images_paths))  # leukemia

      hem_paths.extend(hem_images_paths)
      hem_labels.extend([0] * len(hem_images_paths))  # healthy

  # Take equal-sized sample
  sample_size = min(len(hem_paths), len(all_paths))
  all_indices = random.sample(range(len(all_paths)), sample_size)
  hem_indices = random.sample(range(len(hem_paths)), sample_size)

  image_paths = [all_paths[i] for i in all_indices] + [hem_paths[i] for i in hem_indices]
  labels = [all_labels[i] for i in all_indices] + [hem_labels[i] for i in hem_indices]

  return image_paths, labels



In [75]:

X_path,y = get_paths(TRAIN_DIR)
train_paths, test_paths, train_labels, test_labels = train_test_split(
    X_path, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [84]:
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

class LeukemiaDataset(Dataset):
  def __init__(self, image_paths, labels, img_size=224, augment=False):
      self.image_paths = image_paths
      self.labels = labels
      self.augment = augment

      # Base transform: resize + to tensor
      base_transforms = [
          transforms.Resize((img_size, img_size)),
          transforms.ToTensor(),
          transforms.Normalize([0.485, 0.456, 0.406],  # ImageNet mean/std
                                [0.229, 0.224, 0.225])
      ]

      if self.augment:
          # Add augmentation transforms
          aug_transforms = [
              transforms.RandomHorizontalFlip(),
              transforms.RandomVerticalFlip(),
              transforms.RandomRotation(15),
              transforms.ColorJitter(brightness=0.1, contrast=0.1)
          ]
          self.transform = transforms.Compose(aug_transforms + base_transforms)
      else:
          self.transform = transforms.Compose(base_transforms)

  def __len__(self):
      return len(self.image_paths)

  def __getitem__(self, idx):
      img = Image.open(self.image_paths[idx]).convert("RGB")
      img = self.transform(img)
      label = self.labels[idx]
      return img, label

In [86]:

train_dataset = LeukemiaDataset(train_paths, train_labels, img_size=224, augment=True)
test_dataset  = LeukemiaDataset(test_paths, test_labels, img_size=224, augment=False)
train_loader = DataLoader(train_dataset, batch_size=32, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, num_workers=2)

In [87]:
class LeukemiaCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2,2)

        # After 3 poolings on 128×128:
        # 128 -> 64 -> 32 -> 16
        self.fc1 = nn.Linear(128 * 28 * 28, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [88]:
def train_model(train_loader, test_loader, epochs=35, lr=1e-4):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using", device)

    model = LeukemiaCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)

        # Eval
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                preds = model(imgs).argmax(dim=1)
                correct += (preds == lbls).sum().item()
                total += lbls.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        acc = correct / total
        print(f"Epoch {epoch+1}/{epochs} | Loss={epoch_loss:.4f} | Test Acc={acc:.4f}")

    return model

In [90]:
model = train_model(train_loader,test_loader,epochs=20, lr=1e-4)

Using cuda


Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b1aa03e8180><function _MultiProcessingDataLoaderIter.__del__ at 0x7b1aa03e8180>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
        self._shutdown_workers()self._shutdown_workers()

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        if w.is_alive():if w.is_alive():
 
             ^^^^^^^^^^^^^^^^^^^^^^^
^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
        assert self.

Epoch 1/35 | Loss=0.6461 | Test Acc=0.7515
Epoch 2/35 | Loss=0.5147 | Test Acc=0.7795
Epoch 3/35 | Loss=0.4748 | Test Acc=0.7883
Epoch 4/35 | Loss=0.4689 | Test Acc=0.7987
Epoch 5/35 | Loss=0.4402 | Test Acc=0.8016
Epoch 6/35 | Loss=0.4266 | Test Acc=0.7957
Epoch 7/35 | Loss=0.4067 | Test Acc=0.7839
Epoch 8/35 | Loss=0.3999 | Test Acc=0.8016
Epoch 9/35 | Loss=0.3865 | Test Acc=0.8289
Epoch 10/35 | Loss=0.3790 | Test Acc=0.7618
Epoch 11/35 | Loss=0.3748 | Test Acc=0.8319
Epoch 12/35 | Loss=0.3684 | Test Acc=0.8370
Epoch 13/35 | Loss=0.3582 | Test Acc=0.7109
Epoch 14/35 | Loss=0.3568 | Test Acc=0.8326
Epoch 15/35 | Loss=0.3465 | Test Acc=0.8311
Epoch 16/35 | Loss=0.3463 | Test Acc=0.8525
Epoch 17/35 | Loss=0.3454 | Test Acc=0.8525
Epoch 18/35 | Loss=0.3366 | Test Acc=0.8414
Epoch 19/35 | Loss=0.3416 | Test Acc=0.8518
Epoch 20/35 | Loss=0.3315 | Test Acc=0.8252
Epoch 21/35 | Loss=0.3348 | Test Acc=0.8083
Epoch 22/35 | Loss=0.3264 | Test Acc=0.8444
Epoch 23/35 | Loss=0.3112 | Test Acc=0.85

In [91]:
import torchvision.models as models


In [102]:
device = "cuda" if torch.cuda.is_available() else "cpu"
epochs = 20

model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

# Optional: freeze convolutional layers
'''
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True
'''

optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


In [103]:
import torchvision.models as models


for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

    # Eval
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss={epoch_loss:.4f} | Test Acc={acc:.4f}")

Epoch 1/20 | Loss=0.6468 | Test Acc=0.6984
Epoch 2/20 | Loss=0.5835 | Test Acc=0.6991
Epoch 3/20 | Loss=0.5765 | Test Acc=0.7021
Epoch 4/20 | Loss=0.5949 | Test Acc=0.7279
Epoch 5/20 | Loss=0.5708 | Test Acc=0.7271
Epoch 6/20 | Loss=0.5557 | Test Acc=0.7625
Epoch 7/20 | Loss=0.5657 | Test Acc=0.7588
Epoch 8/20 | Loss=0.5535 | Test Acc=0.7375
Epoch 9/20 | Loss=0.5470 | Test Acc=0.7212
Epoch 10/20 | Loss=0.6158 | Test Acc=0.7235
Epoch 11/20 | Loss=0.5543 | Test Acc=0.7640
Epoch 12/20 | Loss=0.5730 | Test Acc=0.7153
Epoch 13/20 | Loss=0.5938 | Test Acc=0.7220
Epoch 14/20 | Loss=0.5904 | Test Acc=0.7743
Epoch 15/20 | Loss=0.5273 | Test Acc=0.7249
Epoch 16/20 | Loss=0.5734 | Test Acc=0.7625
Epoch 17/20 | Loss=0.5496 | Test Acc=0.7257
Epoch 18/20 | Loss=0.5727 | Test Acc=0.7279
Epoch 19/20 | Loss=0.5807 | Test Acc=0.7330
Epoch 20/20 | Loss=0.5664 | Test Acc=0.7382


In [104]:
model = models.efficientnet_b0(pretrained=True)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # higher LR for classifier
criterion = nn.CrossEntropyLoss()



/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 178MB/s]


In [105]:
epochs = 15
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

    # Eval
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss={epoch_loss:.4f} | Test Acc={acc:.4f}")

# After a few epochs, unfreeze last 2 blocks of EfficientNet
for param in model.features[-2:].parameters():
    param.requires_grad = True

# Lower LR for fine-tuning backbone
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

Epoch 1/15 | Loss=0.4192 | Test Acc=0.6364
Epoch 2/15 | Loss=0.3269 | Test Acc=0.8798
Epoch 3/15 | Loss=0.2913 | Test Acc=0.8938
Epoch 4/15 | Loss=0.2575 | Test Acc=0.8289
Epoch 5/15 | Loss=0.2346 | Test Acc=0.9071
Epoch 6/15 | Loss=0.2203 | Test Acc=0.9181
Epoch 7/15 | Loss=0.2035 | Test Acc=0.9226
Epoch 8/15 | Loss=0.1828 | Test Acc=0.9255
Epoch 9/15 | Loss=0.1793 | Test Acc=0.8894
Epoch 10/15 | Loss=0.1673 | Test Acc=0.9100
Epoch 11/15 | Loss=0.1535 | Test Acc=0.9366
Epoch 12/15 | Loss=0.1463 | Test Acc=0.9159
Epoch 13/15 | Loss=0.1405 | Test Acc=0.9063
Epoch 14/15 | Loss=0.1366 | Test Acc=0.9425
Epoch 15/15 | Loss=0.1249 | Test Acc=0.9100
